# Weak evidence at $p \approx 0.05$

Set up a Monte Carlo experiment with two competing hypotheses about a treatment effect $\theta$:

- $H_0\colon \theta = 0$ (no effect),
- $H_1\colon \theta \sim U[0,1]$ (some non-negative effect, magnitude unknown),

each carrying prior weight $\tfrac{1}{2}$. The analyst observes $\hat{x} \mid \theta \sim N(\theta, \sigma_{\bar x}^2)$ with known standard error $\sigma_{\bar x} = 0.5$, computes the one-sided $z$-statistic $\hat{x} / \sigma_{\bar x}$ and the corresponding $p$-value $1 - \Phi(z)$, and rejects $H_0$ at the 5% level whenever $p < 0.05$.

**Question.** Among trials with $p$ *just barely* below 5% (say $|p - 0.05| \le 0.002$), how often is $H_0$ actually true?

If a $p$-value at the conventional 5% threshold were "strong evidence" against $H_0$, the answer should be very close to zero. We will see that it is not.

In [1]:
import numpy as np
import pandas as pd
from scipy.stats import norm

rng = np.random.default_rng(42)

## Setup: draw $\theta$, draw $\hat x$, compute $z$ and $p$

In [2]:
M = 100_000
sigma_bar_x = 0.5

# Step 1: prior over theta. With probability 1/2, theta = 0 (H_0); else theta ~ U[0,1] (H_1).
H1 = rng.random(M) >= 0.5
theta = np.where(H1, rng.random(M), 0.0)

# Step 2: observation under each hypothesis.
x_hat = rng.normal(theta, sigma_bar_x)

# Step 3: one-sided z-stat and p-value (testing H_0: theta = 0 against theta > 0).
z_stat = x_hat / sigma_bar_x
p_value = 1 - norm.cdf(z_stat)

data = pd.DataFrame({"theta": theta, "x_hat": x_hat, "z_stat": z_stat, "p_value": p_value})
data.head()

,theta,x_hat,z_stat,p_value
0,0.938873,1.084221,2.168442,0.015063
1,0.000000,0.089592,0.179185,0.428896
2,0.536007,0.224074,0.448149,0.327023
3,0.250865,0.723325,1.446650,0.073997
4,0.000000,-0.061352,-0.122703,0.548829


## Filter on barely-significant trials, $|p - 0.05| \le 0.002$

Among the trials whose $p$-value lands in this narrow window, how many came from $H_0$ vs $H_1$? The ratio gives the empirical posterior odds.

In [3]:
barely_sig = np.abs(data["p_value"] - 0.05) <= 0.002

# theta == 0 exactly identifies H_0 (continuous H_1 draws are 0 with probability zero).
n_H0 = ((data["theta"] == 0) & barely_sig).sum()
n_H1 = ((data["theta"] >  0) & barely_sig).sum()
n_total = barely_sig.sum()

post_p_H0 = n_H0 / n_total
post_odds_H1_to_H0 = n_H1 / n_H0

pd.Series({
    "trials with |p - 0.05| <= 0.002":  n_total,
    "  ... from H_0 (theta = 0)":       n_H0,
    "  ... from H_1 (theta > 0)":       n_H1,
    "posterior P(H_0 | barely sig)":    post_p_H0,
    "posterior odds H_1 : H_0":         post_odds_H1_to_H0,
})

trials with |p - 0.05| <= 0.002    757.000000
  ... from H_0 (theta = 0)         188.000000
  ... from H_1 (theta > 0)         569.000000
posterior P(H_0 | barely sig)        0.248349
posterior odds H_1 : H_0             3.026596
dtype: float64